# Análise da Tabela Reservas (FATO)
Iniciaremos o tratamento da tabela que em outros programas será a central das relações. 
Algumas inconsistências dessa base:

|Tipo de inconsistência | tratamento realizado|
|---|---|
|colunas com datas | datetime ao invés de object|
| id_canal com nulos | Manter os nulos — reservas antigas sem canal registrado |
| id_canal tipo object | Converter para Int64 (inteiro que aceita nulos) |
|colunas id_canal e avaliacao_hospedes | verificar o motivo de menos registros|
|coluna avaliacao_hospedes | modificar o tipo para float (decimal)|
|coluna avaliacao_hospedes | modicicar valores discrepantes (acima de 10 e negativos)|


Realizar outras limpezas de acordo com o Comunicado Oficial COM-2025-047, disponibilizado pelo gestor da  empresa em 27 de maio de 2026. Como informado, o tratamento precisa seguir critérios para que haja performance em bancos de dados relacionais.

In [1]:
import pandas as pd

C:\Users\gabriel.goitia\AppData\Roaming\Python\Python313\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [2]:
df_reservas = pd.read_csv('reservas.csv', sep=';', decimal=',')

In [3]:
df_reservas.head()

,id_reserva,id_unidade,id_tipo_quarto,id_cliente,id_canal,data_checkin,data_checkout,qtd_diarias,num_hospedes,avaliacao_hospede,status_reserva,forma_pagamento
0,1,5,1,3,2.0,2024-04-28,2024-04-30,2,1,4.0,confirmada,Cartão de Crédito
1,2,5,1,261,4.0,2023-09-28,2023-10-01,3,2,1.0,concluída,PIX
2,3,10,4,49,4.0,2023-08-01,2023-08-06,5,1,9.0,concluída,Cartão de Crédito
3,4,6,2,23,2.0,2023-02-18,2023-02-21,3,1,9.0,cancelada,PIX
4,5,7,1,42,1.0,2023-01-12,2023-01-19,7,1,6.0,confirmada,Transferência


In [4]:
df_reservas.info()

<class 'pandas.DataFrame'>
RangeIndex: 2457 entries, 0 to 2456
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   id_reserva         2457 non-null   int64
 1   id_unidade         2457 non-null   int64
 2   id_tipo_quarto     2457 non-null   int64
 3   id_cliente         2457 non-null   int64
 4   id_canal           2345 non-null   str  
 5   data_checkin       2457 non-null   str  
 6   data_checkout      2457 non-null   str  
 7   qtd_diarias        2457 non-null   int64
 8   num_hospedes       2457 non-null   int64
 9   avaliacao_hospede  2120 non-null   str  
 10  status_reserva     2457 non-null   str  
 11  forma_pagamento    2457 non-null   str  
dtypes: int64(6), str(6)
memory usage: 344.9 KB


In [5]:
df_reservas['status_reserva'].unique()

<ArrowStringArray>
[ 'confirmada',   'concluída',   'cancelada',     'no-show', 'CONFIRMADA ',
  'Confirmada',       'conf.',  'Concluída ',   'Cancelada',   'Concluida',
     'cancel.',   'CANCELADA',     'NO-SHOW']
Length: 13, dtype: str

Após a indentificação de diferentes nomes dados com a a mesma finalidade na coluna `*status_reserva*`, o grupo decidiu padronizar a nomenclatura:

|Valor encontrado | Padrão oficial |
|---|---|
|confirmada, Confirmada, CONFIRMADA , conf. | Confirmada |
|concluída, Concluída (com espaço) , Concluida | Concluída |
|cancelada, Cancelada, CANCELADA, cancel. | Cancelada |
|no-show, NO-SHOW | No-Show |

In [6]:
mapa_status = {
    'confirmada'  : 'Confirmada',
    'Confirmada'  : 'Confirmada',
    'CONFIRMADA ' : 'Confirmada',
    'conf.' : 'Confirmada',
    'concluída' : 'Concluída',
    'Concluída': 'Concluída',
    'Concluida' : 'Concluída',
    'Concluída ' : 'Concluída',
    'cancelada' : 'Cancelada',
    'Cancelada' : 'Cancelada',
    'CANCELADA' : 'Cancelada',
    'cancel.' : 'Cancelada',
    'no-show' : 'No-Show',
    'NO-SHOW' : 'No-Show'
}

In [7]:
df_reservas['status_reserva'] = df_reservas['status_reserva'].replace(mapa_status) 

In [8]:
df_reservas['status_reserva'].unique()    # mostra os valores distintos


<ArrowStringArray>
['Confirmada', 'Concluída', 'Cancelada', 'No-Show']
Length: 4, dtype: str

In [9]:
df_reservas['status_reserva'].value_counts()  # mostra quantas vezes cada valor aparece

status_reserva
Confirmada    1748
Cancelada      363
Concluída      220
No-Show        126
Name: count, dtype: int64

In [10]:
df_reservas['forma_pagamento'].unique() # relembrando que unique() mostra os valores distintos. Faremos novamente após o tratamento dos dados.

<ArrowStringArray>
['Cartão de Crédito',               'PIX',     'Transferência',
          'Dinheiro',  'Cartão de Débito',              'cred',
               'Pix',               'pix',    'cartao credito',
     'transferencia',          'dinheiro',                'CC',
               'déb',          'DINHEIRO',               'ted',
        'C. Crédito',                'CD',              'cash',
     'cartao debito',               'TED']
Length: 20, dtype: str

In [11]:
mapa_forma_pagto = {
    #'Cartão de Crédito' : 'Cartão de Crédito',
    #'PIX' : 'PIX',
    #'Transferência' : 'Transferência',
    #'Dinheiro' : 'Dinheiro',
    #'Cartão de Débito' : 'Cartão de Débito', 
    'cred' : 'Cartão de Crédito',
    'Pix' : 'PIX',
    'pix' : 'PIX',
    'cartao credito' : 'Cartão de Crédito',
    'transferencia' : 'Transferência',
    'dinheiro' : 'Dinheiro',
    'CC' : 'Cartão de Crédito',
    'déb' : 'Cartão de Débito',
    'DINHEIRO' : 'Dinheiro',
    'ted' : 'Transferência',
    'C. Crédito' : 'Cartão de Crédito',
    'CD' : 'Cartão de Débito',
    'cash' : 'Dinheiro',
    'cartao debito' : 'Cartão de Débito',
    'TED' : 'Transferência'
}

As 5 (cinco) primeiras modalidades do mapa_forma_pagto foram mantidas para fins didáticos, mas podem ser excluídas dessa formação, pois seus valores 'chave : valor' do dicionário estão corretos.

In [12]:
df_reservas['forma_pagamento'] = df_reservas['forma_pagamento'].replace(mapa_forma_pagto) 

In [13]:
df_reservas['forma_pagamento'].unique() # relembrando que unique() mostra os valores distintos

<ArrowStringArray>
['Cartão de Crédito', 'PIX', 'Transferência', 'Dinheiro', 'Cartão de Débito']
Length: 5, dtype: str

In [14]:
df_reservas['forma_pagamento'].value_counts()

forma_pagamento
Dinheiro             511
Cartão de Débito     504
PIX                  496
Cartão de Crédito    474
Transferência        472
Name: count, dtype: int64

Podemos observar que a quantidade de utilização de cada forma de pagamento está bastante equilibrada, nenhuma domina com folga. Uma leitura interessante tendo em vista que 'Dinheiro' e 'Transferência' aparecem de forma equilibrada com as demais. 
Agora, o tratamento dos dados será direcionado a datas, para possíveis estudos de sazonalidades ou importações para outros aplicativos como Power BI e My SQL. Vamos rever que datas estão categorizadas como object e serão mudadas para datetime.

In [15]:
df_reservas.info()

<class 'pandas.DataFrame'>
RangeIndex: 2457 entries, 0 to 2456
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   id_reserva         2457 non-null   int64
 1   id_unidade         2457 non-null   int64
 2   id_tipo_quarto     2457 non-null   int64
 3   id_cliente         2457 non-null   int64
 4   id_canal           2345 non-null   str  
 5   data_checkin       2457 non-null   str  
 6   data_checkout      2457 non-null   str  
 7   qtd_diarias        2457 non-null   int64
 8   num_hospedes       2457 non-null   int64
 9   avaliacao_hospede  2120 non-null   str  
 10  status_reserva     2457 non-null   str  
 11  forma_pagamento    2457 non-null   str  
dtypes: int64(6), str(6)
memory usage: 345.3 KB


In [16]:
df_reservas['data_checkin'] = pd.to_datetime(df_reservas['data_checkin'])

In [17]:
df_reservas['data_checkout'] = pd.to_datetime(df_reservas['data_checkout'])

In [18]:
df_reservas.info()

<class 'pandas.DataFrame'>
RangeIndex: 2457 entries, 0 to 2456
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   id_reserva         2457 non-null   int64         
 1   id_unidade         2457 non-null   int64         
 2   id_tipo_quarto     2457 non-null   int64         
 3   id_cliente         2457 non-null   int64         
 4   id_canal           2345 non-null   str           
 5   data_checkin       2457 non-null   datetime64[us]
 6   data_checkout      2457 non-null   datetime64[us]
 7   qtd_diarias        2457 non-null   int64         
 8   num_hospedes       2457 non-null   int64         
 9   avaliacao_hospede  2120 non-null   str           
 10  status_reserva     2457 non-null   str           
 11  forma_pagamento    2457 non-null   str           
dtypes: datetime64[us](2), int64(6), str(4)
memory usage: 297.3 KB


Agora analisaremos a coluna id_canal:

In [19]:
df_reservas['id_canal'].unique()

<ArrowStringArray>
['2.0', '4.0', '1.0', '3.0', nan]
Length: 5, dtype: str

Os valores estão como '2.0', '4.0' —> texto com decimal. E tem NaN entre os valores. Se tentar converter direto para int64 vai dar erro por causa dos nulos. Primeiro converte-se para float.

In [20]:
df_reservas['id_canal'] = df_reservas['id_canal'].astype(float)

In [21]:
df_reservas['id_canal'].unique()

array([ 2.,  4.,  1.,  3., nan])

In [22]:
df_reservas['id_canal'].isnull().sum()

np.int64(112)

In [23]:
df_reservas[df_reservas['avaliacao_hospede'].isnull()]['status_reserva'].value_counts()

status_reserva
Confirmada    230
Cancelada      54
Concluída      31
No-Show        22
Name: count, dtype: int64

Como visto, foi um achado interessante termos achado `230` valores nulos no status `Confirmada`. Entretanto, não há como o cliente avaliar se ele ainda não se hospedou e portanto não avaliou... Como relatado no Manual de Boas Práticas seguiremos:
"Obs.: Registros com valor nulo neste campo são válidos e não devem ser removidos — representam hóspedes que optaram por não avaliar."

In [24]:
df_reservas['avaliacao_hospede'].unique()

<ArrowStringArray>
[ '4.0',  '1.0',  '9.0',  '6.0', '10.0',    nan,  '2.0',  '8.0',  '7.0',
 '11.0',  '3.0',  '0.0', '-3.0']
Length: 13, dtype: str

In [25]:
df_reservas.info()

<class 'pandas.DataFrame'>
RangeIndex: 2457 entries, 0 to 2456
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   id_reserva         2457 non-null   int64         
 1   id_unidade         2457 non-null   int64         
 2   id_tipo_quarto     2457 non-null   int64         
 3   id_cliente         2457 non-null   int64         
 4   id_canal           2345 non-null   float64       
 5   data_checkin       2457 non-null   datetime64[us]
 6   data_checkout      2457 non-null   datetime64[us]
 7   qtd_diarias        2457 non-null   int64         
 8   num_hospedes       2457 non-null   int64         
 9   avaliacao_hospede  2120 non-null   str           
 10  status_reserva     2457 non-null   str           
 11  forma_pagamento    2457 non-null   str           
dtypes: datetime64[us](2), float64(1), int64(6), str(3)
memory usage: 290.1 KB


-> Valores iguais a zero, a número negativo ou valor superior à nota máxima precisam ser substituídos. Seguiremos o padrão do Manual de Boas Práticas na coluna de avaliação dos hóspedes (vamos filtrar a coluna id_reservas e substituir na coluna avaliacao_hospede): 

In [26]:
'''df_reservas.loc[df_reservas['id_reserva'] == 37, 'avaliacao_hospede'] = 8
df_reservas.loc[df_reservas['id_reserva'] == 194, 'avaliacao_hospede'] = 2
df_reservas.loc[df_reservas['id_reserva'] == 902, 'avaliacao_hospede'] = 10
df_reservas.loc[df_reservas['id_reserva'] == 933, 'avaliacao_hospede'] = 7
df_reservas.loc[df_reservas['id_reserva'] == 1538, 'avaliacao_hospede'] = 3
df_reservas.loc[df_reservas['id_reserva'] == 1718, 'avaliacao_hospede'] = 4
df_reservas.loc[df_reservas['id_reserva'] == 1866, 'avaliacao_hospede'] = 3
df_reservas.loc[df_reservas['id_reserva'] == 2073, 'avaliacao_hospede'] = 9
df_reservas.loc[df_reservas['id_reserva'] == 2167, 'avaliacao_hospede'] = 5
df_reservas.loc[df_reservas['id_reserva'] == 2210, 'avaliacao_hospede'] = 2
'''

"df_reservas.loc[df_reservas['id_reserva'] == 37, 'avaliacao_hospede'] = 8\ndf_reservas.loc[df_reservas['id_reserva'] == 194, 'avaliacao_hospede'] = 2\ndf_reservas.loc[df_reservas['id_reserva'] == 902, 'avaliacao_hospede'] = 10\ndf_reservas.loc[df_reservas['id_reserva'] == 933, 'avaliacao_hospede'] = 7\ndf_reservas.loc[df_reservas['id_reserva'] == 1538, 'avaliacao_hospede'] = 3\ndf_reservas.loc[df_reservas['id_reserva'] == 1718, 'avaliacao_hospede'] = 4\ndf_reservas.loc[df_reservas['id_reserva'] == 1866, 'avaliacao_hospede'] = 3\ndf_reservas.loc[df_reservas['id_reserva'] == 2073, 'avaliacao_hospede'] = 9\ndf_reservas.loc[df_reservas['id_reserva'] == 2167, 'avaliacao_hospede'] = 5\ndf_reservas.loc[df_reservas['id_reserva'] == 2210, 'avaliacao_hospede'] = 2\n"

In [27]:
df_reservas.loc[df_reservas['id_reserva'].isin([37, 194, 902, 933, 1538, 1718, 1866, 2073, 2167, 2210]), 'avaliacao_hospede']

36      11.0
193      0.0
901     11.0
932     -3.0
1537     0.0
1717    -3.0
1865     0.0
2072    11.0
2166    -3.0
2209     0.0
Name: avaliacao_hospede, dtype: str

In [28]:
df_reservas['avaliacao_hospede'] = df_reservas['avaliacao_hospede'].astype(float)

In [29]:
df_reservas['avaliacao_hospede'].dtypes

dtype('float64')

In [30]:
df_reservas.loc[df_reservas['qtd_diarias'] <= 0]

,id_reserva,id_unidade,id_tipo_quarto,id_cliente,id_canal,data_checkin,data_checkout,qtd_diarias,num_hospedes,avaliacao_hospede,status_reserva,forma_pagamento
727,728,4,4,120,1.0,2023-09-22,2023-09-29,0,4,NaN,Confirmada,Cartão de Crédito
954,955,7,3,211,2.0,2023-07-12,2023-07-16,-1,1,9.0,Confirmada,Dinheiro
1133,1134,5,1,125,4.0,2023-07-08,2023-07-11,-1,2,2.0,Cancelada,Transferência
1151,1152,5,2,59,4.0,2023-10-15,2023-10-17,0,1,3.0,Confirmada,Transferência
1447,1448,8,3,52,1.0,2024-11-01,2024-11-07,-1,3,8.0,Confirmada,Dinheiro
1580,1581,10,3,280,2.0,2024-03-12,2024-03-16,0,3,10.0,Concluída,Transferência
2311,2312,8,2,138,4.0,2024-08-25,2024-08-29,0,1,6.0,Confirmada,Dinheiro
2383,2384,5,2,64,2.0,2024-10-01,2024-10-03,-1,2,4.0,Concluída,Dinheiro


In [31]:
df_reservas.loc[df_reservas['id_reserva'] == 728, 'qtd_diarias'] = 4
df_reservas.loc[df_reservas['id_reserva'] == 955, 'qtd_diarias'] = 4
df_reservas.loc[df_reservas['id_reserva'] == 1134, 'qtd_diarias'] = 2
df_reservas.loc[df_reservas['id_reserva'] == 1152, 'qtd_diarias'] = 5
df_reservas.loc[df_reservas['id_reserva'] == 1448, 'qtd_diarias'] = 2
df_reservas.loc[df_reservas['id_reserva'] == 1581, 'qtd_diarias'] = 2
df_reservas.loc[df_reservas['id_reserva'] == 2312, 'qtd_diarias'] = 2
df_reservas.loc[df_reservas['id_reserva'] == 2384, 'qtd_diarias'] = 2

In [32]:
df_reservas.loc[df_reservas['qtd_diarias'] <= 0]

,id_reserva,id_unidade,id_tipo_quarto,id_cliente,id_canal,data_checkin,data_checkout,qtd_diarias,num_hospedes,avaliacao_hospede,status_reserva,forma_pagamento


In [33]:
df_reservas.loc[df_reservas['num_hospedes'] < 0]

,id_reserva,id_unidade,id_tipo_quarto,id_cliente,id_canal,data_checkin,data_checkout,qtd_diarias,num_hospedes,avaliacao_hospede,status_reserva,forma_pagamento
567,568,8,4,199,1.0,2023-07-16,2023-07-23,7,-1,9.0,Confirmada,Cartão de Crédito
674,675,2,5,226,2.0,2023-10-16,2023-10-18,2,-2,10.0,No-Show,Cartão de Débito
902,903,6,3,207,2.0,2024-05-15,2024-05-18,3,-1,6.0,Confirmada,PIX
1200,1201,6,3,120,3.0,2024-11-11,2024-11-12,1,-2,6.0,Confirmada,Dinheiro
1882,1883,3,3,2,4.0,2024-01-15,2024-01-18,3,-2,4.0,Cancelada,Cartão de Crédito
2426,2427,2,2,280,2.0,2023-06-20,2023-06-24,4,-2,9.0,Confirmada,Transferência


In [34]:
df_reservas['num_hospedes'].unique()

array([ 1,  2,  3,  4, -1, -2])

In [35]:
df_reservas.loc[df_reservas['id_reserva'] == 568, 'num_hospedes'] = 2
df_reservas.loc[df_reservas['id_reserva'] == 675, 'num_hospedes'] = 3
df_reservas.loc[df_reservas['id_reserva'] == 903, 'num_hospedes'] = 2
df_reservas.loc[df_reservas['id_reserva'] == 1201, 'num_hospedes'] = 3
df_reservas.loc[df_reservas['id_reserva'] == 1883, 'num_hospedes'] = 1
df_reservas.loc[df_reservas['id_reserva'] == 2427, 'num_hospedes'] = 1

In [36]:
df_reservas.loc[df_reservas['num_hospedes'] < 0]

,id_reserva,id_unidade,id_tipo_quarto,id_cliente,id_canal,data_checkin,data_checkout,qtd_diarias,num_hospedes,avaliacao_hospede,status_reserva,forma_pagamento


In [37]:
df_reservas.info()

<class 'pandas.DataFrame'>
RangeIndex: 2457 entries, 0 to 2456
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   id_reserva         2457 non-null   int64         
 1   id_unidade         2457 non-null   int64         
 2   id_tipo_quarto     2457 non-null   int64         
 3   id_cliente         2457 non-null   int64         
 4   id_canal           2345 non-null   float64       
 5   data_checkin       2457 non-null   datetime64[us]
 6   data_checkout      2457 non-null   datetime64[us]
 7   qtd_diarias        2457 non-null   int64         
 8   num_hospedes       2457 non-null   int64         
 9   avaliacao_hospede  2120 non-null   float64       
 10  status_reserva     2457 non-null   str           
 11  forma_pagamento    2457 non-null   str           
dtypes: datetime64[us](2), float64(2), int64(6), str(2)
memory usage: 283.3 KB


## Conclusão — Tratamento da Tabela Reservas (Fato)

A tabela `reservas.csv` passou por um processo de auditoria e limpeza seguindo os critérios estabelecidos pela **NaraHoteis — Rede Hoteleira 
do Estado do Rio de Janeiro**, conforme o documento oficial **COM-2025-047**, de 27 de maio de 2025.

### Tratamentos realizados

| Campo | Problema identificado | Tratamento aplicado |
|---|---|---|
| `status_reserva` | Variações de escrita | Padronizado para: Confirmada, Cancelada, No-Show, Concluída |
| `forma_pagamento` | Abreviações inconsistentes | Padronizado para: Cartão de Crédito, Cartão de Débito, PIX, Dinheiro, Transferência |
| `data_checkin` | Tipo object | Convertido para datetime64 |
| `data_checkout` | Tipo object | Convertido para datetime64 |
| `id_canal` | Tipo object com nulos | Convertido para float64 — nulos mantidos (reservas antigas) |
| `avaliacao_hospede` | Valores fora da escala + tipo object | 10 valores corrigidos conforme COM-2025-047, convertido para float64 — nulos mantidos (hóspedes que optaram por não avaliar) |
| `qtd_diarias` | Valores 0 e -1 | 8 valores corrigidos conforme COM-2025-047 |
| `num_hospedes` | Valores negativos | 6 valores corrigidos conforme COM-2025-047 |

A base tratada será exportada como `reservas_limpa.csv` e estará pronta para uso nas etapas de análise estatística, banco de dados e painel gerencial.

In [45]:
df_reservas.to_csv('reservas_limpa.csv', index=False, sep=';', decimal=',')

In [42]:
df_reservas.isna().sum()

id_reserva             0
id_unidade             0
id_tipo_quarto         0
id_cliente             0
id_canal             112
data_checkin           0
data_checkout          0
qtd_diarias            0
num_hospedes           0
avaliacao_hospede    337
status_reserva         0
forma_pagamento        0
dtype: int64

In [ ]:
dff = df_reservas.groupby('id_unidade').size().reset_index(name='teste')
dff


,id_unidade,teste
0,1,186
1,2,184
2,3,261
3,4,214
4,5,231
5,6,197
6,7,195
7,8,181
8,9,208
9,10,215


In [44]:
df_reservas['id_canal'] = df_reservas['id_canal'].fillna(5)